# WaterSoftHack 2026 — Day 1: Cloud Computing (Hands-On)

**Welcome!** You are looking at a Jupyter notebook, but the computer running it is **not your laptop**.
It is a virtual machine on **Jetstream2**, a research cloud in Indiana, paid for with our project's
NSF **ACCESS** credits. You reached it with nothing but a browser and a login. That is cloud computing,
and by the end of this hour you will have used it for real water science.

## The story we are building (all three days)

We are building a **real-time flood early-warning system** for a network of USGS stream gauges.

- **Today (Cloud):** pull *live* streamflow data, train a short-term forecast, then scale it to many
  gauges at once.
- **Tomorrow (Edge):** move the smarts out to the sensors themselves, under real field limits.
- **Thursday (Hybrid):** combine the two into one pipeline.

## What you will do in the next hour

1. See exactly where your code is running (spoiler: the cloud).
2. Pull **live** river data from the USGS national gauge network.
3. Store it the way cloud systems do.
4. Train a **streamflow now-cast** that predicts the next hour of flow.
5. **Scale out** to many gauges at once and watch the cloud's real superpower kick in.
6. Learn how to *not* waste our shared credits.

> No prior cloud experience needed. Run each cell with **Shift+Enter**. Read the short notes between cells.

---
## Part 0 — Wait, where is this code actually running?

Your browser is on your laptop. But the **kernel** (the thing that runs the Python) lives on a
remote machine in the cloud. Run the cell below and look at the answers. The hostname, the number of
CPU cores, the operating system: none of that is your laptop. It is a computer you were handed, on
demand, over the internet.

> **Concept - the cloud is rented computing.** A "cloud" is just someone else's computers in a big data center that you rent by the hour over the internet, the way you draw electricity from a grid instead of running your own generator. When you open this notebook your browser is on your laptop, but the **kernel** (the program that actually runs the Python) lives on a rented Linux machine in Indiana called a **virtual machine (VM)**. Its **CPU cores** do the computing and its **RAM** holds data while you work. You did not buy or install any of it, and when class ends we hand it back. That is the whole idea: compute is a utility you rent and return, not a box you own.

In [ ]:
import os, sys, platform

print("Python version   :", sys.version.split()[0])
print("Machine hostname :", platform.node())
print("Operating system :", platform.system(), platform.release())
print("CPU cores here   :", os.cpu_count())
print("Your home folder :", os.path.expanduser("~"))

# How much memory did the cloud give us? (Linux-only; skipped if unavailable.)
try:
    with open("/proc/meminfo") as f:
        total_kb = int(f.readline().split()[1])
    print("RAM available    :", round(total_kb / 1e6, 1), "GB")
except Exception:
    print("RAM available    : (couldn't read /proc/meminfo on this OS)")

**Think about it.** You did not buy this machine, install an OS, or plug in any RAM. You asked for a
computer and got one. When class ends, we hand it back. *That* is the cloud idea: compute is a utility
you rent by the hour, like electricity, not a box you own.

---
## Part 1 — Pull live water data from the cloud

The USGS runs a national network of stream gauges and publishes their readings through a free web API
(the *National Water Information System*, NWIS). We will ask it for **discharge** (river flow, USGS
parameter code `00060`, in cubic feet per second) at the **Iowa River at Iowa City** gauge — a few
minutes from campus.

The helper below fetches real data. If the network hiccups during class, it **falls back to a
realistic synthetic gauge** so nothing breaks. Either way the rest of the notebook works.

**The synthetic hydrograph (the fallback gauge).** When the live fetch fails, discharge in cfs is generated over $n=24\,\text{days}$ hourly steps $t=0,\dots,n-1$ as a slow baseflow drift, a daily cycle, three Gaussian rain pulses, and Gaussian noise, floored at 1:

$$Q(t)=\max\!\Big(1,\ \underbrace{900+120\sin\tfrac{1.5\,t}{n}}_{\text{baseflow drift}}+\underbrace{30\sin\tfrac{2\pi t}{24}}_{\text{daily cycle}}+\underbrace{\sum_{c\,\in\,\{0.25n,\,0.6n,\,0.85n\}}\!\!\!500\,e^{-\frac{1}{2}\left(\frac{t-c}{36}\right)^{2}}}_{\text{three rain events}}+\ \varepsilon_t\Big),\qquad \varepsilon_t\sim\mathcal{N}(0,\,12^{2}).$$

The three Gaussian bumps sit at a quarter, three-fifths, and 0.85 of the record, each about $500$ cfs tall with a spread of 36 hours.

In [ ]:
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def synthetic_streamflow(site, days=14, seed=0):
    # A believable, smoothly varying hydrograph: baseflow, a daily cycle, a few gentle rain bumps.
    rng = np.random.default_rng(abs(hash(site)) % (2**32) if seed == 0 else seed)
    n = days * 24
    idx = pd.date_range(end=pd.Timestamp.now("UTC").floor("h"), periods=n, freq="1h")
    t = np.arange(n)
    base = 900 + 120 * np.sin(t / (n / 1.5))        # slow drift over the record
    daily = 30 * np.sin(2 * np.pi * t / 24)         # daily cycle
    bumps = np.zeros(n)
    for center in (int(n * 0.25), int(n * 0.6), int(n * 0.85)):
        bumps += 500 * np.exp(-0.5 * ((t - center) / 36) ** 2)   # gentle rain events
    flow = np.clip(base + daily + bumps + rng.normal(0, 12, n), 1, None)
    return pd.DataFrame({"discharge_cfs": flow}, index=idx)

def fetch_streamflow(site, days=14):
    # Returns (DataFrame indexed by time, site_name, source_string).
    url = "https://waterservices.usgs.gov/nwis/iv/"
    params = {"format": "json", "sites": site, "parameterCd": "00060",
              "period": f"P{days}D", "siteStatus": "all"}
    try:
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        series = r.json()["value"]["timeSeries"]
        if not series:
            raise ValueError("no data returned")
        values = series[0]["values"][0]["value"]
        name = series[0]["sourceInfo"]["siteName"]
        d = pd.DataFrame(values)
        d["dateTime"] = pd.to_datetime(d["dateTime"])
        d["discharge_cfs"] = pd.to_numeric(d["value"], errors="coerce")
        d = d[d["discharge_cfs"] >= 0].set_index("dateTime")[["discharge_cfs"]]
        d = d.resample("1h").mean().interpolate().dropna()
        if len(d) < 48:
            raise ValueError("too few points")
        return d, name, "USGS live"
    except Exception as e:
        return synthetic_streamflow(site, days), f"Synthetic gauge {site}", f"synthetic fallback ({type(e).__name__})"

IOWA_CITY = "05454500"
df, name, source = fetch_streamflow(IOWA_CITY, days=14)

print("Gauge :", name)
print("Source:", source)
print("Rows  :", len(df), "hourly readings")
df.tail()

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(df.index, df["discharge_cfs"], lw=1.8)
plt.title(f"Live streamflow - {name}")
plt.ylabel("Discharge (cfs)")
plt.xlabel("Time (UTC)")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

That curve is a **hydrograph**. Every bump is real water moving down a real river, delivered to a
computer in Indiana, and drawn on your screen — with about ten lines of code and zero infrastructure
of your own. Try changing `IOWA_CITY` to another 8-digit USGS site number later and re-run.

---
## Part 2 — Where does the data live? Cloud storage

On your laptop you save files to a hard drive. The cloud gives you two flavors of storage:

- **Block storage** — behaves like a normal disk attached to your VM (what your home folder uses here).
- **Object storage** — a giant web-accessible bucket of files (Amazon calls it *S3*; OpenStack calls it
  *Swift*). This is how huge public datasets are shared. For example, the entire **National Water Model**
  forecast archive lives in public cloud object storage that anyone can read.

For now we will save our data to the VM's disk, then read it back — the everyday version of the same idea.

> **Concept - bring your code to the data.** Big water datasets no longer fit on a laptop. The National Water Model archive alone is petabytes. The cloud flips the old habit: instead of downloading data to your computer, you run your computer next to the data. Cloud storage comes in two flavors. **Block storage** behaves like the drive in your laptop, fast and attached to one machine, which is what your home folder uses here. **Object storage** is a giant web-accessible bucket that scales almost without limit and that anyone you allow can read, which is how the National Water Model and NOAA radar are published. You read only the slice you need, in the cloud, without ever moving the whole archive.

In [ ]:
from pathlib import Path

workspace = Path.home() / "watersofthack_day1_cloud"
workspace.mkdir(exist_ok=True)

csv_path = workspace / f"streamflow_{IOWA_CITY}.csv"
df.to_csv(csv_path)
print("Saved:", csv_path)

# Parquet is the columnar format cloud data lakes prefer; save one too if the library is present.
try:
    parquet_path = workspace / f"streamflow_{IOWA_CITY}.parquet"
    df.to_parquet(parquet_path)
    print("Saved:", parquet_path, "(compact columnar format)")
except Exception as e:
    print("(Parquet skipped:", type(e).__name__, "- CSV is enough for today)")

reloaded = pd.read_csv(csv_path, index_col=0, parse_dates=True)
print("Reloaded", len(reloaded), "rows. Matches original:", len(reloaded) == len(df))

---
## Part 3 — Train a streamflow *now-cast*

A **now-cast** predicts the very near future — here, **next hour's flow** from the last several hours.
It is the core of a flood early-warning system: if the model says flow is about to spike, you warn people.

One subtlety makes the difference between a model that works and one that embarrasses you. River flow
trends up and down over weeks, and a tree-based model cannot predict a value it never saw in training.
So instead of predicting the flow *level* directly, we predict the **hourly change** and add it back to
the current flow. The change is stationary and well-behaved, which makes the forecast robust.

The recipe:

- **Features:** current flow, the last 6 hourly values (`lag1`..`lag6`), a 6-hour average, and hour of day.
- **Target:** the **change** in flow over the next hour (then reconstruct the level).
- **Split by time:** train on the earlier 70%, test on the most recent 30% (never shuffle a time series).
- **Model:** a Random Forest — robust and needs no tuning to work.
- **Scores:** **RMSE** (typical error, in cfs) and **NSE** (Nash-Sutcliffe Efficiency — the hydrologist's
  favorite: 1.0 is perfect, 0 means "no better than predicting the average").

> **Concept - how the forecast is scored, and why we predict the change.** Two numbers grade a forecast. **RMSE** (root-mean-square error) is the typical miss in the same units as the data, here cubic feet per second, so smaller is better. **NSE** (Nash-Sutcliffe Efficiency) is the hydrologist's favorite: 1.0 is perfect, 0 means the model is no better than always guessing the long-run average, and a negative value means it is worse than that. We split the record by **time**, training on the earlier part and testing on the most recent part, because shuffling a time series would let the model peek at the future. And notice the subtlety: we predict the hourly **change** in flow and add it back, not the flow level directly. A tree-based model cannot predict a value higher than anything it saw in training, but the change stays small and well-behaved, so this keeps the forecast honest during a rise.

**The math behind the scores.** With observed flow $Q^{\mathrm{obs}}_t$, forecast $Q^{\mathrm{sim}}_t$ over $n$ test hours, and mean observed flow $\bar{Q}^{\mathrm{obs}}$:

$$\mathrm{RMSE}=\sqrt{\frac{1}{n}\sum_{t=1}^{n}\left(Q^{\mathrm{sim}}_t-Q^{\mathrm{obs}}_t\right)^2},\qquad \mathrm{NSE}=1-\frac{\sum_{t=1}^{n}\left(Q^{\mathrm{sim}}_t-Q^{\mathrm{obs}}_t\right)^2}{\sum_{t=1}^{n}\left(Q^{\mathrm{obs}}_t-\bar{Q}^{\mathrm{obs}}\right)^2}.$$

RMSE is an absolute error in cfs. NSE measures the model against the "always predict the mean" baseline, so $\mathrm{NSE}=1$ is perfect and $\mathrm{NSE}=0$ only ties that baseline. The delta method predicts the hourly change and rebuilds the level:

$$\Delta_t=Q_{t+h}-Q_t,\qquad \hat{Q}_{t+h}=Q_t+\hat{\Delta}_t.$$

In [ ]:
from sklearn.ensemble import RandomForestRegressor

def make_features(frame, n_lags=6, horizon=1):
    d = frame.rename(columns={"discharge_cfs": "flow"}).copy()
    for lag in range(1, n_lags + 1):
        d[f"lag{lag}"] = d["flow"].shift(lag)
    d["roll6"] = d["flow"].rolling(6).mean()
    d["hour"] = d.index.hour
    d["future"] = d["flow"].shift(-horizon)    # flow horizon hours ahead
    d["delta"] = d["future"] - d["flow"]       # the change we actually predict
    return d.dropna()

def nse(obs, sim):
    obs, sim = np.asarray(obs), np.asarray(sim)
    return 1 - np.sum((sim - obs) ** 2) / np.sum((obs - obs.mean()) ** 2)

FEATURES = ["flow"] + [f"lag{l}" for l in range(1, 7)] + ["roll6", "hour"]

feat = make_features(df)
split = int(len(feat) * 0.7)
train, test = feat.iloc[:split], feat.iloc[split:]

model = RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=0)
model.fit(train[FEATURES], train["delta"])

pred_future = test["flow"].values + model.predict(test[FEATURES])   # reconstruct the level
obs_future = test["future"].values

rmse = float(np.sqrt(np.mean((pred_future - obs_future) ** 2)))
print(f"RMSE : {rmse:,.1f} cfs")
print(f"NSE  : {nse(obs_future, pred_future):.3f}   (closer to 1.0 is better)")

In [ ]:
plt.figure(figsize=(11, 4))
plt.plot(test.index, obs_future, label="Observed next-hour flow", lw=2)
plt.plot(test.index, pred_future, label="Now-cast prediction", lw=2, ls="--")
plt.title(f"Streamflow now-cast on held-out data - {name}")
plt.ylabel("Discharge (cfs)")
plt.xlabel("Time (UTC)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

You just trained a machine-learning model **on cloud hardware**. It never touched your laptop. And if this
model were bigger, say a deep neural network, you could shut down this plain CPU machine and boot one with
an **NVIDIA A100 GPU** on Jetstream2 in a couple of minutes, for a few more credits per hour. Renting the
*right* hardware for the moment, then giving it back, is the whole game.

---
## Part 4 — Scale out: the cloud's real superpower

One gauge was easy. But a real flood system watches **thousands** of gauges. The National Water Model
forecasts **2.7 million** river reaches. You cannot do that one at a time.

First we train a now-cast for **ten real Iowa-basin gauges** at once. Then we simulate a much larger
network to see the cloud's defining trick: **elasticity** — ask for many CPU cores and finish in a
fraction of the time.

> **Concept - elasticity is the cloud's superpower.** One gauge was easy. A real flood system watches thousands. **Elasticity** means you rent a lot of computers for the hour you need them and give them back afterward. Two ways to grow: **scale up** swaps in a bigger machine, and **scale out** runs many machines, or many CPU cores, in parallel. The `joblib` library below spreads the gauge training across every core in this VM at once, so a job that took minutes on one core finishes in seconds. You pay only for the hours you actually use, which is why renting beats owning a machine that would sit idle between storms.

In [ ]:
sites = [
    "05454500",  # Iowa River at Iowa City
    "05453100",  # Iowa River at Marengo
    "05465500",  # Iowa River at Wapello
    "05464500",  # Cedar River at Cedar Rapids
    "05457700",  # Cedar River at Charles City
    "05458000",  # Little Cedar River near Ionia
    "05420500",  # Mississippi River at Clinton
    "05451500",  # Iowa River at Marshalltown
    "05481950",  # Beaver Creek near Grimes
    "05490600",  # Des Moines River at St. Francisville
]

print("Fetching", len(sites), "real gauges ...")
site_data = {s: fetch_streamflow(s, days=14) for s in sites}

def train_site(site, data, n_estimators=300):
    frame, site_name, src = data
    try:
        f = make_features(frame)
        cut = int(len(f) * 0.7)
        tr, te = f.iloc[:cut], f.iloc[cut:]
        # single-threaded model, so parallelism can happen ACROSS gauges (a fair timing test)
        m = RandomForestRegressor(n_estimators=n_estimators, n_jobs=1, random_state=0)
        m.fit(tr[FEATURES], tr["delta"])
        level = te["flow"].values + m.predict(te[FEATURES])
        return {"site": site, "name": site_name[:34], "source": src.split()[0],
                "n": len(f), "NSE": round(nse(te["future"].values, level), 3)}
    except Exception:
        return {"site": site, "name": site_name[:34], "source": "error", "n": 0, "NSE": float("nan")}

nse_table = pd.DataFrame([train_site(s, site_data[s]) for s in sites])
print(nse_table.to_string(index=False))

Ten rivers, ten forecasts, most with a solid NSE. Now let's feel **scale**. We simulate a larger network
of gauges and train a now-cast for every one — first **sequentially** on one core, then **in parallel**
across every core in this VM.

In [ ]:
import time
from joblib import Parallel, delayed

N_GAUGES = 60
sim_data = {f"sim{i:03d}": (synthetic_streamflow(f"sim{i:03d}", days=21, seed=i + 1),
                            f"sim{i:03d}", "synthetic") for i in range(N_GAUGES)}

t0 = time.perf_counter()
_ = [train_site(s, sim_data[s], n_estimators=400) for s in sim_data]
t_seq = time.perf_counter() - t0

t0 = time.perf_counter()
_ = Parallel(n_jobs=-1)(delayed(train_site)(s, sim_data[s], 400) for s in sim_data)
t_par = time.perf_counter() - t0

print(f"Trained now-casts for {N_GAUGES} simulated gauges")
print(f"  Sequential (1 core)     : {t_seq:6.1f} s")
print(f"  Parallel  ({os.cpu_count():>2} cores)  : {t_par:6.1f} s")
print(f"  Speedup                 : {t_seq / t_par:5.1f}x")

**That gap is the point.** Same code, same machine, but by using all the cores at once the job finished
several times faster. And on Jetstream2 you are not stuck with this VM's cores: you could launch a
64-core machine for one hour (about 64 credits), forecast an entire state's gauge network before lunch,
then shelve it. Compute expands to fit the problem and shrinks back when you are done.

---
## Part 5 — Be a good cloud citizen (this is real money)

Our project has **200,000 ACCESS credits**, and on Jetstream2 **1 credit = 1 CPU core running for 1 hour**.
This whole workshop costs about a hundred credits. But credits are shared and real, so three habits matter:

- **Shelve your VM when you are done.** A *running* VM costs credits every hour; a *shelved* one costs **zero**.
  Leaving a 32-core VM on overnight by accident burns ~350 credits for nothing.
- **Right-size the machine.** Do not rent 64 cores to edit text. Match the machine to the task.
- **Ask for a GPU only when you will use it.** GPUs cost more per hour. Boot one, do the training, shelve it.

Cloud computing is renting. The bill is the resources you leave running, not the code you write.

> **Concept - the bill is what you leave running.** On Jetstream2 the meter is simple: one credit is one CPU core running for one hour. Our project holds 200,000 credits and this whole workshop costs about a hundred. The catch is that a running VM bills every hour whether or not you are using it, so an idle machine left on overnight is pure waste. The habit that protects the shared budget is to **shelve** the machine when you finish: a shelved VM costs zero and your files are preserved. Right-size the machine to the task, and rent a GPU only for the hour you actually train on it.

---
## Wrap-up

In one hour, entirely on the cloud, you:

- ran code on a rented computer in another state,
- pulled **live** data from the national river-gauge network,
- stored and reloaded it the cloud way,
- trained a **streamflow now-cast** and scored it with RMSE and NSE,
- **scaled out** across gauges and measured the speedup,
- and learned how to keep the bill sane.

### Looking ahead to tomorrow (Edge)

Notice what we did today: we shipped **every raw reading** to the cloud and did all the thinking there.
That is fine when the gauge sits next to fiber internet. But most gauges are in remote places, running on a
solar panel and a weak cell signal. Sending everything is slow, expensive, and dies when the signal drops.

**Tomorrow we flip it around:** put a little bit of intelligence *inside the sensor* so it decides what is
worth sending. That is **edge computing**, and we will simulate a whole fleet of smart gauges.

---
## Optional challenges (if you finish early)

1. **Different river.** Change `IOWA_CITY` to a USGS gauge near your own university
   (find one at https://waterdata.usgs.gov/) and re-run Parts 1–3.
2. **Longer horizon.** In `make_features`, call it with `horizon=3` (3 hours ahead). Does NSE drop?
   Why is forecasting further out harder?
3. **Different model.** Swap `RandomForestRegressor` for
   `from sklearn.ensemble import GradientBoostingRegressor`. Compare NSE.
4. **More history.** Increase `n_lags` to 12. Does more past help the forecast?